In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the dataset
# (Adjust path if your data is located in a different subdirectory)
df = pd.read_csv('../data/insurance_data.csv')

print(f"📊 Initial Shape of Dataset: {df.shape}")

# =========================================================================
# 2. DATA CLEANING & IMPUTATION
# =========================================================================
# Check for any lingering missing values across features
missing_summary = df.isnull().sum()
print("\n🔍 Missing value counts per column:")
print(missing_summary[missing_summary > 0] if missing_summary.sum() > 0 else "None found! Clear to proceed.")

# Pro-tip: If any rows have missing structural targets or variables, 
# we fill numerics with the median and categories with the mode.
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

# =========================================================================
# 3. ADVANCED FEATURE ENGINEERING
# =========================================================================
# Let's build structural risk indicators that help tree models capture patterns
df['Income_to_Risk_Ratio'] = df['AnnualIncome'] / (df['RiskScore'] + 1)
df['Deductible_Ratio'] = df['Deductible'] / (df['AnnualPremium'] + 1)

# Clean up structural string issues
df['Province'] = df['Province'].astype(str).str.strip()
df['Gender'] = df['Gender'].astype(str).str.strip()
df['VehicleType'] = df['VehicleType'].astype(str).str.strip()

# =========================================================================
# 4. FILTERING FOR THE SEVERITY MODEL TARGET
# =========================================================================
# CRITICAL ACTUARIAL STEP: Severity models ONLY look at rows where an actual claim happened!
severity_df = df[df['TotalClaims'] > 0].copy()
print(f"\n✂️ Isolate Severity Sub-segment (TotalClaims > 0): {severity_df.shape}")

# Define our modeling feature matrix (X) and target vector (y)
feature_cols = [
    'Age', 'AnnualIncome', 'RiskScore', 'Deductible', 'NCD', 
    'Income_to_Risk_Ratio', 'Deductible_Ratio', 'Gender', 'Province', 'VehicleType'
]

X = severity_df[feature_cols]
y = severity_df['TotalClaims'] # Predicting the financial liability size

# =========================================================================
# 5. CATEGORICAL ENCODING & TRAIN/TEST SPLIT
# =========================================================================
# Turn text columns (Gender, Province, VehicleType) into 1s and 0s safely
X_encoded = pd.get_dummies(X, drop_first=True)

# Split into train/test sets using an 80:20 distribution
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

print(f"\n🚀 Data Splitting Matrix Complete:")
print(f" -> Training Features Shape: {X_train.shape}")
print(f" -> Testing Features Shape:  {X_test.shape}")

📊 Initial Shape of Dataset: (10000, 21)

🔍 Missing value counts per column:
None found! Clear to proceed.

✂️ Isolate Severity Sub-segment (TotalClaims > 0): (1535, 23)

🚀 Data Splitting Matrix Complete:
 -> Training Features Shape: (1228, 15)
 -> Testing Features Shape:  (307, 15)


In [14]:
!pip install xgboost shap

In [15]:
import sys
sys.path.append('../')
from src.modeling import train_and_evaluate_regressors

# Train all three algorithms using our split data matrix
metrics_table, trained_suite = train_and_evaluate_regressors(X_train, X_test, y_train, y_test)

print("\n=========================================================")
print("         RISK SEVERITY MODEL COMPARISON TABLE            ")
print("=========================================================")
print(metrics_table)
print("=========================================================\n")


         RISK SEVERITY MODEL COMPARISON TABLE            
                        RMSE  R² Score
Linear Regression  5236.8237    0.2246
Random Forest      5108.1277    0.2623
XGBoost            5255.3130    0.2191



In [16]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Clean object/categorical data right here in the notebook cell
X_train_numeric = X_train.apply(pd.to_numeric, errors='coerce').fillna(0).astype('float64')
X_test_numeric = X_test.apply(pd.to_numeric, errors='coerce').fillna(0).astype('float64')

# 2. Extract your champion XGBoost model from the trained_suite dictionary
# Note: If your dictionary key is exactly "XGBoost", make sure this string matches it!
best_model_name = "XGBoost"  
best_model = trained_suite[best_model_name]

print(f"🚀 Generating SHAP feature importance for {best_model_name}...")

# 3. CRITICAL FIX: Pass the new NUMERIC variables into the function!
explainer, shap_vals = generate_shap_explainer(best_model, X_train_numeric, X_test_numeric, best_model_name)

print("✅ SHAP analysis complete! You are ready to run the plotting cell below.")

🚀 Generating SHAP feature importance for XGBoost...
✅ SHAP analysis complete! You are ready to run the plotting cell below.


In [ ]:
# Generate the SHAP Summary Plot
plt.title(f"SHAP Feature Importance Summary: {best_model_name}", fontsize=14)
shap.summary_plot(shap_vals, X_test_numeric, plot_type="bar")
plt.show()